> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# CSR network, PCSF and GNN demonstration

VIGOR 2.12.1. The small graph is synthetic and demonstrates APIs. Replace it with curated variant-to-gene and interaction evidence for biological interpretation.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
# A small demonstration network; these names are not HS-rat gene evidence.
from scipy.sparse.csgraph import connected_components
edges=pd.DataFrame({"source":["geneA","geneB","geneC","geneD","geneE","geneA"],
                    "target":["geneB","geneC","geneD","geneE","geneF","geneF"],"cost":[1.,1.,1.,1.,1.,2.]})
prizes={"geneA":2.,"geneF":1.5}
adjacency,nodes,node_index=vigor.edge_table_to_csr(edges)
print("CSR components:",connected_components(adjacency,directed=False)[0])
result=vigor.prioritize_network(edges,prizes,method="greedy")
print("PCSF nodes:",result.nodes)
print("diffusion:",vigor.diffuse_scores(edges,prizes))
features=pd.DataFrame(np.random.default_rng(42).normal(size=(6,3)),index=nodes)
labels=pd.Series([0,1,0,1,0,1],index=nodes)
loci=pd.Series(["l1","l1","l2","l2","l3","l3"],index=nodes)
gnn=vigor.fit_heterogeneous_gnn(edges,features,labels,
    split={"held_out_loci":["l3"]},node_loci=loci,architecture="gcn",epochs=20,patience=5,random_state=42)
print("GNN:",type(gnn).__name__)
